# Calculating Median Percentile Shifts Across Signatures and Factors by Perturbed Gene and by Perturbed Guide

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys 
import scanpy as sc 
import muon as mu

sys.path.append('../utils')

import signature_heatmaps as signature_heatmaps
import factor_labels as factor_labels


## Load Data

In [ ]:
data_dir = "<path to processed data>"

cite_6tf_path = os.path.join(data_dir, "cite_6tf_cleaned_revisions.h5mu")
cite_imgl_path = os.path.join(data_dir, "cite_imgl_cleaned_revisions.h5mu")
merged_6tf_path = os.path.join(data_dir, "adata_revisions_merged_6tf.h5ad")

In [ ]:
mdata_dict = {}
mdata_dict['cite_6tf'] = mu.read_h5mu(cite_6tf_path)
mdata_dict['cite_imgl'] = mu.read_h5mu(cite_imgl_path)

adata_dict = {}
adata_dict['merged_6tf'] = sc.read_h5ad(merged_6tf_path)
adata_dict['cite_6tf'] = mdata_dict['cite_6tf'].mod['rna'].copy()
adata_dict['cite_imgl'] = mdata_dict['cite_imgl'].mod['rna'].copy()

In [ ]:
print(mdata_dict['cite_6tf'])
print(mdata_dict['cite_imgl'])
print(adata_dict['merged_6tf'])

In [ ]:
signature_cols_ordered = ['homeostatic_score_ucell',
 'interferon_score_ucell',
 'chemokine_score_ucell',
 'antigen_presenting_score_ucell',
 'dam_score_ucell',
 'lipid_dam_score_ucell']

## Masking for analysis
to exclude ntc_g5 + foxk1_g2 + mixscale_cutoff >= 0

In [ ]:
guides_to_exclude = ['FOXK1_g2', 'non-targeting_g5']

adata_6tf_clean = adata_dict['merged_6tf'][~adata_dict['merged_6tf'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_imgl_clean = adata_dict['cite_imgl'][~adata_dict['cite_imgl'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_6tf_clean.shape, adata_imgl_clean.shape

In [ ]:
print(mdata_dict['cite_6tf'].shape, mdata_dict['cite_imgl'].shape)
mdata_6tf_clean = mdata_dict['cite_6tf'][~mdata_dict['cite_6tf'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_imgl_clean = mdata_dict['cite_imgl'][~mdata_dict['cite_imgl'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_6tf_clean.shape, mdata_imgl_clean.shape

In [ ]:
mixscale_col = "mixscale_score"

In [ ]:
adata_6tf_masked = adata_6tf_clean[adata_6tf_clean.obs[mixscale_col] >= 0].copy()
adata_imgl_masked = adata_imgl_clean[adata_imgl_clean.obs[mixscale_col] >= 0].copy()
adata_6tf_masked.shape, adata_imgl_masked.shape

In [ ]:
adata_masked_dict = {}
adata_masked_dict['iTF'] = adata_6tf_masked
adata_masked_dict['iMG'] = adata_imgl_masked

# Genes to Include

In [ ]:
genes_to_include = ['DNMT1', 'IRF9', 'STAT2', 'SMAD3', 'PRDM1', 'ZNF532', 'NTC']
itf_genes = ['DNMT1', 'IRF9', 'STAT2', 'SMAD3']
img_genes = ['PRDM1', 'ZNF532']

# Save Directory

In [ ]:
save_dir = "<path to output directory>"


# Calculate Median Percentile Shifts

### for signatures

In [ ]:
df_long_6tf_signatures = factor_labels.build_df_long(adata_6tf_masked, descriptive_cols= signature_cols_ordered, diff_type_name="Signature")
df_long_imgl_signatures = factor_labels.build_df_long(adata_imgl_masked, descriptive_cols= signature_cols_ordered, diff_type_name="Signature")


In [ ]:
percentile_df_6tf_sign_guide = factor_labels.calculate_median_percentile_shifts(df_long_6tf_signatures, factor_col="Signature", guide_col = "perturbed_guide")
percentile_df_imgl_sign_guide = factor_labels.calculate_median_percentile_shifts(df_long_imgl_signatures, factor_col="Signature", guide_col = "perturbed_guide")


In [ ]:
percentile_df_imgl_sign_guide.to_csv(save_dir + "ps_iMG_signature_guide_v5.csv", index=None)
percentile_df_6tf_sign_guide.to_csv(save_dir + "ps_iTF_signature_guide_v5.csv", index=None)

In [ ]:
percentile_df_6tf_sign_gene = factor_labels.calculate_median_percentile_shifts(df_long_6tf_signatures, factor_col="Signature", guide_col = "perturbed_gene")
percentile_df_imgl_sign_gene = factor_labels.calculate_median_percentile_shifts(df_long_imgl_signatures, factor_col="Signature", guide_col = "perturbed_gene")


In [ ]:
percentile_df_imgl_sign_gene.to_csv(save_dir + "ps_iMG_signature_gene_v5.csv", index=None)
percentile_df_6tf_sign_gene.to_csv(save_dir + "ps_iTF_signature_gene_v5.csv", index=None)



### for factors

In [ ]:
factor_cols = adata_6tf_masked.obs.columns[(adata_6tf_masked.obs.columns.str.startswith("f")) &
                                           ~(adata_6tf_masked.obs.columns.isin(['f6','f12', 'f13']))]


In [ ]:
factor_cols_labels = [factor_labels.get_direct_factor_map()[f] for f in factor_cols]

In [ ]:
adata_6tf_masked.obs = adata_6tf_masked.obs.rename(columns=factor_labels.get_direct_factor_map())
adata_imgl_masked.obs = adata_imgl_masked.obs.rename(columns=factor_labels.get_direct_factor_map())

In [ ]:
df_long_6tf_fact = factor_labels.build_df_long(adata_6tf_masked, descriptive_cols= factor_cols_labels,diff_type_name="Factor")
df_long_imgl_fact = factor_labels.build_df_long(adata_imgl_masked, descriptive_cols= factor_cols_labels,diff_type_name="Factor")


In [ ]:
percentile_df_6tf_fact = factor_labels.calculate_median_percentile_shifts(df_long_6tf_fact, factor_col="Factor", guide_col = "perturbed_guide")
percentile_df_imgl_fact = factor_labels.calculate_median_percentile_shifts(df_long_imgl_fact, factor_col="Factor", guide_col = "perturbed_guide")

In [ ]:
percentile_df_imgl_fact.to_csv(save_dir + "ps_iMG_factor_guide_v5.csv", index=None)
percentile_df_6tf_fact.to_csv(save_dir + "ps_iTF_factor_guide_v5.csv", index=None)

In [ ]:
percentile_df_6tf_fact_gene = factor_labels.calculate_median_percentile_shifts(df_long_6tf_fact, factor_col="Factor", guide_col = "perturbed_gene")
percentile_df_imgl_fact_gene = factor_labels.calculate_median_percentile_shifts(df_long_imgl_fact, factor_col="Factor", guide_col = "perturbed_gene")


In [ ]:
percentile_df_imgl_fact_gene.to_csv(save_dir + "ps_iMG_factor_gene_v5.csv", index=None)
percentile_df_6tf_fact_gene.to_csv(save_dir + "ps_iTF_factor_gene_v5.csv", index=None)